In [1]:
from pinecone import Pinecone
import os
pc = Pinecone(api_key = os.getenv("PINECONE_API_KEY")) 

In [2]:
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(
    model = "qwen3-embedding:8b",
    dimensions = 1024
)

In [3]:
res = embeddings.embed_query("Hello World")

In [4]:
print(f"Dimenssion: {len(res)}")

Dimenssion: 1024


In [5]:
from pinecone import ServerlessSpec

index_name = "rag"

if not pc.has_index(index_name):
  pc.create_index(
      name = index_name,
      dimension = 1024,
      metric = "cosine",
      spec = ServerlessSpec(cloud = "aws", region = "us-east-1")
  )
index = pc.Index(index_name)

In [6]:
from langchain_pinecone import PineconeVectorStore

vector_store = PineconeVectorStore(
    index = index,
    embedding = embeddings,
    text_key = "text"
)

In [7]:
from langchain_core.documents import Document

document_1 = Document(
    page_content="I had chocolate chip pancakes and scrambled eggs for breakfast this morning.",
    metadata={"source": "tweet"},
)

document_2 = Document(
    page_content="The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.",
    metadata={"source": "news"},
)

document_3 = Document(
    page_content="Building an exciting new project with LangChain - come check it out!",
    metadata={"source": "tweet"},
)

document_4 = Document(
    page_content="Robbers broke into the city bank and stole $1 million in cash.",
    metadata={"source": "news"},
)

document_5 = Document(
    page_content="Wow! That was an amazing movie. I can't wait to see it again.",
    metadata={"source": "tweet"},
)

document_6 = Document(
    page_content="Is the new iPhone worth the price? Read this review to find out.",
    metadata={"source": "website"},
)

document_7 = Document(
    page_content="The top 10 soccer players in the world right now.",
    metadata={"source": "website"},
)

document_8 = Document(
    page_content="LangGraph is the best framework for building stateful, agentic applications!",
    metadata={"source": "tweet"},
)

document_9 = Document(
    page_content="The stock market is down 500 points today due to fears of a recession.",
    metadata={"source": "news"},
)

document_10 = Document(
    page_content="I have a bad feeling I am going to get deleted :(",
    metadata={"source": "tweet"},
)

documents = [
    document_1,
    document_2,
    document_3,
    document_4,
    document_5,
    document_6,
    document_7,
    document_8,
    document_9,
    document_10,
]

In [8]:
vector_store.add_documents(documents = documents)

['f1962708-508f-412b-9cbc-4df3bced215e',
 'eac7a887-f3cf-4e61-a2f3-916458cdd520',
 '3ca554f2-1bb0-49ad-9b89-05811e24e370',
 '69287970-d4fd-4ba1-87c3-5015957725d1',
 '4631d5e1-766d-436c-a8e8-b74969981a58',
 '9d47bb2c-38b5-47ea-808c-6f1e022663e8',
 'e9ece7bb-1d11-44c7-b2de-909554799101',
 'ed546b24-151e-4da2-8ec9-5dbd3b4a5d59',
 'b86fc19e-f140-400b-9a52-2c63d63eb4e1',
 'd260a873-2c4a-4d6d-ae02-e7f0b089ee5a']

### Query Directly

In [9]:
results = vector_store.similarity_search(
    "LangChain provides abstractions to make working with LLMs easy",
    k=2,
    filter={"source": "tweet"},
)
for res in results:
    print(f"* {res.page_content} [{res.metadata}]")

* Building an exciting new project with LangChain - come check it out! [{'source': 'tweet'}]
* LangGraph is the best framework for building stateful, agentic applications! [{'source': 'tweet'}]


In [10]:
results = vector_store.similarity_search_with_score(
    "Will it be hot tomorrow?", k=1, filter={"source": "news"}
)
for res, score in results:
    print(f"* [SIM={score:3f}] {res.page_content} [{res.metadata}]")

* [SIM=0.597007] The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees. [{'source': 'news'}]


### Retriever

In [11]:
retriever = vector_store.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"k": 1, "score_threshold": 0.4},
)
retriever.invoke("Stealing from the bank is a crime", filter={"source": "news"})

[Document(id='69287970-d4fd-4ba1-87c3-5015957725d1', metadata={'source': 'news'}, page_content='Robbers broke into the city bank and stole $1 million in cash.')]